Add __pow__ to the Value class so you can compute x ** n. Verify that d/dx(x^3) at x=2 equals 12.0.

In [3]:
class Value:
    def __init__(self,data,children=(),op=''):
        self.data=data
        self.grad=0.0
        self._backward=lambda:None #private/internal — don't touch unless you know what you're doing."
        self._prev=set(children)
        self._op=op
        
    def __repr__(self):
        return f"Value(data={self.data:.4f},grad={self.grad:.4f})"
    
    def __pow__(self, n):
            out = Value(self.data ** n, (self,), f'**{n}')
            def _backward():
                self.grad += n * (self.data ** (n - 1)) * out.grad
            out._backward = _backward
            return out
        
    def backward(self):
        # Topological order all children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # Go one variable at a time and apply chain rule
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [4]:
x=Value(2.0)
y=x**3
y.backward()

print(f"x = {x}")
print(f"y = x^3 = {y}")
print(f"dy/dx at x=2: {x.grad}")
print(f"Expected (3*x^2 = 3*4): 12.0")
print(f"Verification: {'PASS' if abs(x.grad - 12.0) < 1e-6 else 'FAIL'}")


x = Value(data=2.0000,grad=12.0000)
y = x^3 = Value(data=8.0000,grad=1.0000)
dy/dx at x=2: 12.0
Expected (3*x^2 = 3*4): 12.0
Verification: PASS


Add tanh as an activation function. Verify that tanh'(0) = 1 and tanh'(2) = 0.0707 (approx).

In [5]:
import math

class Value:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op
        
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            # d/dx tanh(x) = 1 - tanh(x)^2
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


# ---------- Verification ----------
print("=" * 50)
print("Verifying tanh'(x) = 1 - tanh(x)^2")
print("=" * 50)

# Test 1: tanh'(0) should be 1.0
x1 = Value(0.0)
y1 = x1.tanh()
y1.backward()
print(f"\ntanh'(0):")
print(f"  x.data        = {x1.data}")
print(f"  tanh(x)       = {y1.data}")
print(f"  x.grad        = {x1.grad:.6f}")
print(f"  Expected      = 1.000000")
print(f"  Verification  = {'PASS ✓' if abs(x1.grad - 1.0) < 1e-6 else 'FAIL ✗'}")

# Test 2: tanh'(2) should be approx 0.0707
x2 = Value(2.0)
y2 = x2.tanh()
y2.backward()
print(f"\ntanh'(2):")
print(f"  x.data        = {x2.data}")
print(f"  tanh(2)       = {y2.data:.6f}")
print(f"  x.grad        = {x2.grad:.6f}")
print(f"  Expected      = 0.070650")
print(f"  Verification  = {'PASS ✓' if abs(x2.grad - 0.07065) < 1e-4 else 'FAIL ✗'}")

Verifying tanh'(x) = 1 - tanh(x)^2

tanh'(0):
  x.data        = 0.0
  tanh(x)       = 0.0
  x.grad        = 1.000000
  Expected      = 1.000000
  Verification  = PASS ✓

tanh'(2):
  x.data        = 2.0
  tanh(2)       = 0.964028
  x.grad        = 0.070651
  Expected      = 0.070650
  Verification  = PASS ✓


Build a computation graph for a single neuron: y = relu(w1x1 + w2x2 + b). Compute all five gradients and verify against PyTorch.

In [6]:
import math
import random

# ---------- Autograd Engine ----------
class Value:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(children)
        self._op = op
        
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out
    
    def __radd__(self, other):        # other + self
        return self + other
    
    def __rmul__(self, other):        # other * self
        return self * other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __pow__(self, n):
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            # gradient flows only where input > 0
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


# ---------- Build the neuron: y = relu(w1*x1 + w2*x2 + b) ----------
random.seed(42)
x1 = Value(2.0)
x2 = Value(0.0)
w1 = Value(-3.0)
w2 = Value(1.0)
b  = Value(6.8813735870195432)

# Forward pass
n = w1*x1 + w2*x2 + b   # pre-activation
y = n.relu()            # output

print("Forward pass:")
print(f"  x1 = {x1.data}, x2 = {x2.data}")
print(f"  w1 = {w1.data}, w2 = {w2.data}, b = {b.data}")
print(f"  n  = w1*x1 + w2*x2 + b = {n.data:.4f}")
print(f"  y  = relu(n)           = {y.data:.4f}")

# Backward pass
y.backward()

print("\nGradients (our engine):")
print(f"  dy/dw1 = {w1.grad:.6f}")
print(f"  dy/dw2 = {w2.grad:.6f}")
print(f"  dy/db  = {b.grad:.6f}")
print(f"  dy/dx1 = {x1.grad:.6f}")
print(f"  dy/dx2 = {x2.grad:.6f}")


Forward pass:
  x1 = 2.0, x2 = 0.0
  w1 = -3.0, w2 = 1.0, b = 6.881373587019543
  n  = w1*x1 + w2*x2 + b = 0.8814
  y  = relu(n)           = 0.8814

Gradients (our engine):
  dy/dw1 = 2.000000
  dy/dw2 = 0.000000
  dy/db  = 1.000000
  dy/dx1 = -3.000000
  dy/dx2 = 1.000000


In [7]:
import torch

tx1 = torch.tensor(2.0, requires_grad=True)
tx2 = torch.tensor(0.0, requires_grad=True)
tw1 = torch.tensor(-3.0, requires_grad=True)
tw2 = torch.tensor(1.0, requires_grad=True)
tb  = torch.tensor(6.8813735870195432, requires_grad=True)

tn = tw1*tx1 + tw2*tx2 + tb
ty = torch.relu(tn)
ty.backward()

print("\nGradients (PyTorch):")
print(f"  dy/dw1 = {tw1.grad.item():.6f}")
print(f"  dy/dw2 = {tw2.grad.item():.6f}")
print(f"  dy/db  = {tb.grad.item():.6f}")
print(f"  dy/dx1 = {tx1.grad.item():.6f}")
print(f"  dy/dx2 = {tx2.grad.item():.6f}")

# ---------- Side-by-side comparison ----------
print("\n" + "="*62)
print(f"{'grad':<8}{'ours':>14}{'pytorch':>14}{'match':>14}")
print("="*62)
pairs = [
    ('dy/dw1', w1.grad, tw1.grad.item()),
    ('dy/dw2', w2.grad, tw2.grad.item()),
    ('dy/db',  b.grad,  tb.grad.item()),
    ('dy/dx1', x1.grad, tx1.grad.item()),
    ('dy/dx2', x2.grad, tx2.grad.item()),
]
for name, ours, torch_v in pairs:
    match = "✓" if abs(ours - torch_v) < 1e-6 else "✗"
    print(f"{name:<8}{ours:>14.6f}{torch_v:>14.6f}{match:>14}")
print("="*62)


Gradients (PyTorch):
  dy/dw1 = 2.000000
  dy/dw2 = 0.000000
  dy/db  = 1.000000
  dy/dx1 = -3.000000
  dy/dx2 = 1.000000

grad              ours       pytorch         match
dy/dw1        2.000000      2.000000             ✓
dy/dw2        0.000000      0.000000             ✓
dy/db         1.000000      1.000000             ✓
dy/dx1       -3.000000     -3.000000             ✓
dy/dx2        1.000000      1.000000             ✓
